# Reproducing Fujimoto, Fujimoto, Elorette et al. (2026) from dandiset 001785

**Paper.** Fujimoto S. H., Fujimoto A., Elorette C. et al. "Deep brain stimulation induces white
matter remodeling and functional changes to brain-wide networks." *Nature Neuroscience* (2026),
doi:10.1038/s41593-026-02301-4.

**Dataset.** DANDI:001785, version 0.260615.1835 (doi:10.48324/dandi.001785/0.260615.1835).
The paper cites only its other deposit, DANDI:001773, so the most recent published version of
001785 is pinned. It has the same files as 0.260522.2215 and 13 more electron micrographs than
the four April versions (shown below). No earlier version matches the claims file's description
better.

**Text the claims were written from.** The version of record (Europe PMC, PMC13337501) with the
Supplementary Information, as the claims file `claims/001785/10.1038-s41593-026-02301-4.yaml`
says.

**Outside input.** DANDI:001773 version 0.260416.1939, the paper's own deposit of scripts and
source-data spreadsheets. It supplies labels the dandiset lacks (which voxel belongs to which
tract, CB subregion or target ROI) and the behaviour bins. Each use is tested by a link claim.

| Claim | Status here |
|---|---|
| link-fc-voxels, link-fa-voxels, link-if-counts | tested (link claims) |
| ed1-behavior-increase, ed1-sham-no-behavior-change | tested (data elsewhere, DANDI:001773) |
| fig2a-cb-fa-increase, fig2a-no-other-tract-change, ed2-cb-both-monkeys, fig2b-mcc-cb-increase | tested |
| fig6a-sham-cb-fa-decrease, fig6b-sham-dacc-cb-decrease | tested |
| fig3c-cc1-dapi-ratio-mcc, ed5-cc1-count-dapi, ed6a-dacc-cb-no-change, ed6b-uf-fm-no-change | tested |
| fig5a-pre-scc-connectivity, fig5b-scc-decreases, fig5b-scc-increases, ed7-decreases-both-monkeys, fig5-bilateral-changes | tested |
| fig5c-network-decreases, fig5c-smn-increase, disc-no-motor-roi-change, fig5c-vis-no-change, fig5bc-san-no-net-change | tested |
| fig6d-sham-scc-little-change, fig6e-sham-no-network-reduction | tested |
| fig6c-sham-cc1-no-change | blocked: methods underspecified (the two size filters give opposite outcomes; results shown) |
| fig3f-gratio-decrease | blocked (other): the per-axon measurements were made by hand on axons the raters chose; the dandiset holds the images only |
| fig4a, fig4b, fig4c, fig4-control, ed10c (connectivity matrices) | blocked: data missing (no EPI time series; DANDI:001773 holds only the paper's group matrices, `Source Data FC matrix.xlsx`, and their labels, `FC_ROI_label.xlsx`, results with nothing in 001785 to link them to) |
| fig5a-control-group-no-reduction | blocked: data missing for reading (a), which needs runs; reading (b) is run without the cluster-level correction and shown |

No random numbers are drawn anywhere in this notebook.

In [ ]:
import json
import numpy as np
import pandas as pd
import requests
import fujimoto2026 as h

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
A = h.assets()
print(h.DANDISET, h.VERSION, len(A), "assets,", round(A["size"].sum() / 1e9, 2), "GB")

## Data check

In [ ]:
# File inventory by folder, and the difference from the other published versions.
A["folder"] = A.path.str.split("/").str[:-1].str.join("/")
print(A.groupby("folder").agg(files=("path", "size"), GB=("size", lambda s: round(s.sum() / 1e9, 3))))
for v in ["0.260416.1945", "0.260522.2215"]:
    B = h.assets(h.DANDISET, v)
    extra = sorted(set(A.path) - set(B.path))
    changed = sorted(p for p, b in zip(A.path, A.blob) if p in set(B.path) and b != B.set_index("path").blob[p])
    print(f"against {v}: {len(B)} assets; only in {h.VERSION}: {len(extra)} {extra}; changed: {len(changed)}")

# Files deposited more than once: identical content has the same blob.
dup = A[A.duplicated("blob", keep=False)].sort_values("blob")
print("assets sharing content with another asset:", len(dup))
print(dup[["path", "size"]].to_string() if len(dup) else "")

# dataset_description.json must be JSON (BIDS).
raw = h.fetch("dataset_description.json").decode("utf-8")
try:
    json.loads(raw)
    print("dataset_description.json parses as JSON")
except json.JSONDecodeError as e:
    print("dataset_description.json is not valid JSON:", e)
    print(raw)

In [ ]:
# MRI files: grid, voxel size, value range.
rows = []
for p in A.path[A.path.str.startswith("MRI/")]:
    if "/T1_" in p:
        continue
    img = h.load_nifti(p)
    d = np.asarray(img.dataobj, dtype=float)
    nz = d[d != 0]
    rows.append(dict(path=p, shape=d.shape, voxel_mm=tuple(np.round(img.header.get_zooms(), 3)),
                     units=img.header.get_xyzt_units()[0], nonzero=nz.size,
                     min=nz.min(), max=nz.max(), origin=tuple(np.round(img.affine[:3, 3], 3))))
mri = pd.DataFrame(rows)
print(mri.to_string())
print("FA maps with values above 1:", int((mri[mri.path.str.contains('/FA/')]["max"] > 1).sum()), "of",
      int(mri.path.str.contains('/FA/').sum()))

The FA maps are 86 x 86 x 42 voxels of 1.5 mm, each with its own origin (each session in its
own space). Their maximum is 1.21 to 1.22, above the bound of 1 that FA has by definition; these
are voxels where the tensor fit gives negative eigenvalues, outside the tracts. The FC files
are single volumes on one 43 x 52 x 33 grid of 1.5 mm, in NMT space, one per subject and
session: they are seed maps, not EPI time series. The dandiset holds no tract masks, no ROI
masks, no EPI runs and no cell counts or axon measurements.

In [ ]:
# Seed maps: are they the normalized Fisher z of the paper? The paper's source data list, per
# voxel, its AFNI coordinates and the normalized z ("fc"). Compare with the dandiset's maps.
fc_src = {"NT": h.source_sheet("dbs_FC25norm_mkNT_concatenated_all_data_a.xlsx"),
          "D": h.source_sheet("dbs_FC25norm_mkD_concatenated_all_data_a.xlsx")}
code_of = {1: "N", 2: "T", 3: "D"}
fc_rows = []
for grp, src in fc_src.items():
    src = src.dropna(subset=["fc", "subject"]).copy()
    for (subj, t), s in src.groupby(["subject", "time_condition"]):
        mk = code_of[int(subj)]
        img = h.fc_map(mk, "pre" if t == 1 else "post")
        d = np.asarray(img.dataobj, dtype=float)
        ijk, off = h.rai_to_index(img, s.x, s.y, s.z)
        inside = np.all((ijk >= 0) & (ijk < d.shape), axis=1)
        v = d[tuple(ijk[inside].T)]
        dev = np.abs(v - s.fc.values[inside])
        fc_rows.append(dict(monkey=mk, session="pre" if t == 1 else "post", voxels=len(s),
                            max_offset_from_grid=off, inside=inside.mean(),
                            max_abs_diff=dev.max(), frac_within_1e4=(dev < 1e-4).mean(),
                            map_max=d.max()))
fc_link = pd.DataFrame(fc_rows)
print(fc_link.to_string())

Every voxel the paper's source data list lies on a centre of the dandiset's grid (offset 0) once
the AFNI coordinates are read as RAI (x and y sign-flipped), and the dandiset's value there equals
the paper's normalized z to within 1e-4 (float rounding in the spreadsheet). So the FC files are
the paper's normalized seed maps: each voxel's Fisher z divided by the session's maximum
(Methods, EPI data analysis). The maps' own maxima are below 1, so the maximum used was taken
over voxels outside the deposited volume (or before masking); the files do not say which, and
nothing here depends on it, since the values are the paper's. The file names and the README do
not say this; it is a finding (metadata_insufficient).

In [ ]:
# FA maps against the paper's per-voxel FA values (which have no positions).
def fa_table(name, mkD=False):
    if mkD:
        t = h.source_sheet(name, header=None).iloc[:, :4]
        t.columns = ["FA", "prepost", "side", "part"]
        t["mk"] = 3
    else:
        t = h.source_sheet(name).iloc[:, :5]
        t.columns = ["FA", "mk", "prepost", "side", "part"]
    t = t.dropna(subset=["FA", "mk", "prepost", "side", "part"]).copy()
    for c in ["mk", "prepost", "side", "part"]:
        t[c] = t[c].astype(int)
    return t.reset_index(drop=True)

fa_src = {"whole_NT": fa_table("dbs_FA_wholetract_data_a.xlsx"),
          "sub_NT": fa_table("dbs_FA_CB_subpart_data_a.xlsx"),
          "whole_D": fa_table("dbs_FA_wholetract_mkD_data_a.xlsx", mkD=True),
          "sub_D": fa_table("dbs_FA_CB_subpart_mkD_data_a.xlsx", mkD=True)}
fa_maps = {(mk, s): np.asarray(h.fa_map(mk, s).dataobj, dtype=np.float32)
           for mk in "NTD" for s in ("pre", "post")}
link_rows = []
for key, t in fa_src.items():
    t["voxel"] = -1
    t["nmatch"] = 0
    t["FA_ds"] = np.nan
    t["path"] = "none"
    for (mk, pp), s in t.groupby(["mk", "prepost"]):
        vol = fa_maps[(code_of[mk], "pre" if pp == 1 else "post")]
        idx, n = h.match_values(vol, s.FA.values)
        path = np.where(idx >= 0, "exact", "none").astype(object)
        # values written to the spreadsheet with fewer digits: nearest voxel within 1e-6; n counts
        # every voxel inside that window, so that an ambiguous match is seen
        miss = idx < 0
        if miss.any():
            flat = vol.ravel()
            for j in np.flatnonzero(miss):
                dev = np.abs(flat - s.FA.values[j])
                k = np.argmin(dev)
                if dev[k] < 1e-6:
                    idx[j], n[j], path[j] = k, int((dev < 1e-6).sum()), "nearest"
        t.loc[s.index, "path"] = path
        t.loc[s.index, "voxel"] = idx
        t.loc[s.index, "nmatch"] = n
        t.loc[s.index[idx >= 0], "FA_ds"] = vol.ravel()[idx[idx >= 0]]
    g = t.groupby(["mk", "prepost"])
    link_rows += [dict(table=key, mk=code_of[mk], prepost=pp, values=len(s),
                       matched=(s.voxel >= 0).sum(), exact=(s.path == "exact").sum(),
                       nearest=(s.path == "nearest").sum(), ambiguous=(s.nmatch > 1).sum(),
                       ambiguous_nearest=((s.nmatch > 1) & (s.path == "nearest")).sum(),
                       distinct_voxels=s.voxel[s.voxel >= 0].nunique(),
                       max_abs_diff=np.nanmax(np.abs(s.FA_ds - s.FA)))
                  for (mk, pp), s in g]
fa_link = pd.DataFrame(link_rows)
print(fa_link.to_string())

In [ ]:
# Tract voxels per monkey, side and tract, and whether a voxel is used twice within a tract.
for key, t in fa_src.items():
    print(key)
    print(t[t.voxel >= 0].groupby(["mk", "prepost", "side", "part"]).size().unstack(["prepost"]).to_string())
    rep = t[t.voxel >= 0].groupby(["mk", "prepost", "side", "part"]).voxel.apply(lambda v: v.duplicated().sum())
    print("  repeated voxels within a group:", int(rep.sum()))
    # Is a repeat the paper's (the same value listed twice) or a voxel two different values were
    # both matched to?
    u = t[t.voxel >= 0]
    same_val = u.groupby(["mk", "prepost", "side", "part"]).FA.apply(lambda v: v.duplicated().sum())
    pair = u.groupby(["mk", "prepost", "side", "part", "voxel"]).FA.nunique()
    print("  values listed more than once within a group:", int(same_val.sum()),
          "; voxels matched by two or more different values:", int((pair > 1).sum()))
# Voxels listed in more than one tract or side of a session (not repeats within a group): the same
# voxel in two tract masks, or two different values matched to one voxel? A voxel whose value has
# more than one candidate (nmatch > 1) may stand for two voxels of equal value.
TRACT_OF = {"whole": {1: "CB", 2: "UF", 3: "FM"}, "sub": {1: "SCC", 2: "dACC", 3: "MCC", 4: "PCC"}}
for key, t in fa_src.items():
    u = t[t.voxel >= 0]
    for (mk, pp), s in u.groupby(["mk", "prepost"]):
        g = s.groupby("voxel")
        groups = g.apply(lambda v: tuple(sorted(set(zip(v.side, v.part)))))
        shared = groups[groups.map(len) > 1]
        sides = shared.map(lambda gs: len({sd for sd, _ in gs}))
        names = shared.map(lambda gs: "+".join(f"{'stim' if sd == 1 else 'ctrl'} {TRACT_OF[key.split('_')[0]][pt]}"
                                               for sd, pt in gs))
        print(f"{key} {code_of[mk]} {'pre' if pp == 1 else 'post'}: rows {len(s)}, distinct voxels {s.voxel.nunique()},"
              f" rows beyond the first per voxel {len(s) - s.voxel.nunique()};"
              f" voxels in two or more groups {len(shared)} (across sides {int((sides > 1).sum())},"
              f" listed with different values {int((g.FA.nunique()[shared.index] > 1).sum())},"
              f" with an ambiguous match {int((g.nmatch.max()[shared.index] > 1).sum())})")
        if len(shared):
            print("    ", names.value_counts().to_dict())

In [ ]:
# Immunofluorescence images: name parsing, channels, and the fields of the paper's sheets.
IF = A[A.path.str.startswith("IF/")].copy()
parsed = IF.path.map(h.parse_if_name)
print("names not parsed:", IF.path[parsed.isna()].tolist())
IF = pd.concat([IF.reset_index(drop=True), pd.DataFrame(list(parsed))], axis=1)
IF["region"] = IF.folder.str.replace("IF/IF_CC1_", "", regex=False)
IF["monkey"] = IF.section.str[0]
print(IF.groupby(["region", "monkey", "hemi"]).size().unstack("hemi"))
print("names with a suffix:", IF.loc[IF["mark"] != "", "path"].tolist())
# bytes per channel value from the file size (1024 x 1024 x 3 values per file)
IF["bytes_per_value"] = (IF["size"] / (1024 * 1024 * 3)).round().astype(int)
print("bytes per value by section:", IF.groupby("section").bytes_per_value.first().to_dict())
im = h.load_tiff(IF.path.iloc[0])
print("example", IF.path.iloc[0], im.shape, im.dtype, "channel maxima", im.reshape(-1, 3).max(0))

Each IF file is an RGB image of 1024 x 1024 pixels: red is CC1 (Alexa 555), blue is DAPI, green
is empty. Sections N603, N613 and N623 are 8-bit, the others 16-bit (bytes per value above). The files carry no pixel size (the TIFF
resolution tag is the default 72 per inch). N122 RW has six fields where the design has five.

In [ ]:
# Electron micrographs: hemisphere codes in the names, and the images named in the paper's
# per-axon sheet (DANDI:001773), which holds the hand measurements.
EM = A[A.path.str.startswith("EM/")].copy()
EM["stem"] = EM.path.str.slice(3, -4)
EM["group"] = EM.stem.str.extract(r"^(NI?[LR]|T[LR])")[0]
print(EM.groupby("group").size().to_dict())
em = h.source_sheet("dbs_EMgratio_MCC_CB_data_a.xlsx", header=None)
print("first rows of the per-axon sheet (its header rows and two axons):")
print(em.iloc[:4].to_string())
names = em.iloc[2:, 1].dropna().astype(str).str.strip()
print("axon rows in the sheet:", int(em.iloc[2:, 0].notna().sum()), "; images named:", names.nunique())
print("named images not in the dandiset:", sorted(set(names) - set(EM.stem)))
print("dandiset images not named in the sheet:", len(set(EM.stem) - set(names)), sorted(set(EM.stem) - set(names))[:40])

The 181 micrographs are named by monkey and hemisphere (NIL and NIR for N, TL and TR for T),
block and image. The per-axon sheet of DANDI:001773 (its first rows above) gives, per axon, the
image name and the four measurements, and no position in the image, so which axons were measured
cannot be found in the micrographs. The g-ratio of the paper rests on these hand measurements of
axons the raters chose, which the dandiset does not hold; fig3f-gratio-decrease is blocked (other:
a repeat would need a rater's choices; see the header). TR-1-8r.tif is a byte-identical copy of
TR-1-8.tif.


## link-fc-voxels

> The dandiset's seed maps can be linked, voxel by voxel, to the target ROI voxels listed in the
> paper's source data (DANDI:001773), so that the ROI labels of the source data can be applied to
> the dandiset's values.

The table above (data check) is the link: matched by position.

In [ ]:
print(fc_link[["monkey", "session", "voxels", "max_offset_from_grid", "inside", "max_abs_diff", "frac_within_1e4"]].to_string())
print("all voxels placed on the grid:", bool((fc_link.max_offset_from_grid < 1e-6).all() and (fc_link.inside == 1).all()))
print("fraction of voxels whose value agrees within 1e-4, minimum over maps:", fc_link.frac_within_1e4.min())

**Result: the link holds.** All listed voxels of N, T and D in both sessions fall on grid centres
of the dandiset's maps, and the dandiset's values agree with the paper's normalized z to within
1e-4 at every one of them. The seed analyses below take the ROI label of each voxel from the
source data and its value from the dandiset.

In [ ]:
# Seed-analysis table: ROI labels from the source data, values from the dandiset.
ROI = {1: "PCC", 2: "vmFC", 3: "STG", 4: "insula", 5: "dACC", 6: "DLPFC", 7: "PPC", 8: "area25",
       9: "area32", 10: "MCC", 11: "HIP", 12: "AMY", 13: "sensory", 14: "motor", 15: "SMA",
       16: "V1", 17: "MT"}
NET = {"DMN": [1, 2, 3], "SAN": [4, 5], "CEN": [6, 7], "LIM": [1, 5, 8, 9, 10, 11, 12],
       "SMN": [13, 14, 15], "VIS": [16, 17]}

def seed_table(grp):
    src = fc_src[grp].dropna(subset=["fc", "subject"]).copy()
    parts = []
    for (subj, t), s in src.groupby(["subject", "time_condition"]):
        img = h.fc_map(code_of[int(subj)], "pre" if t == 1 else "post")
        d = np.asarray(img.dataobj, dtype=float)
        ijk, _ = h.rai_to_index(img, s.x, s.y, s.z)
        s = s.copy()
        s["z"] = d[tuple(ijk.T)]          # the dandiset's normalized z
        parts.append(s)
    df = pd.concat(parts, ignore_index=True)
    df = df.rename(columns={"subject": "mk", "time_condition": "pp"})
    for c in ["mk", "pp", "side", "roi"]:
        df[c] = df[c].astype(int)
    return df[["mk", "pp", "side", "roi", "z"]]

seedNT = seed_table("NT")
seedD = seed_table("D")
print("N and T rows:", len(seedNT), " D rows:", len(seedD))
cnt = seedNT[seedNT.pp == 1].groupby(["roi", "side"]).size().unstack("side")
cnt.index = [ROI[i] for i in cnt.index]
cnt.columns = ["stim side", "control side"]
print("voxels per ROI and side, N and T pooled (one session):")
print(cnt.T.to_string())

TERMS4 = [("mk",), ("pp",), ("side",), ("roi",), ("pp", "side"), ("pp", "roi"), ("side", "roi"), ("pp", "side", "roi")]
TERMS3 = [("pp",), ("side",), ("roi",), ("pp", "side"), ("pp", "roi"), ("side", "roi"), ("pp", "side", "roi")]

def network_table(df):
    parts = []
    for i, (name, rois) in enumerate(NET.items(), start=1):
        s = df[df.roi.isin(rois)].copy()
        s["roi"] = i            # the network takes the place of the ROI factor, as in the code
        parts.append(s)
    return pd.concat(parts, ignore_index=True)

def prepost_tests(df, terms, other, nlev, names):
    tab, model = h.anovan(df, "z", ["mk", "pp", "side", "roi"] if other else ["pp", "side", "roi"], terms)
    pairs = [((1, s, r), (2, s, r)) for s in (1, 2) for r in range(1, nlev + 1)]
    res, k, dfe = h.tukey_kramer(df, model, ["pp", "side", "roi"], pairs, other=other)
    res["side"] = ["stim" if a[1] == 1 else "control" for a in res.a]
    res["target"] = [names[a[2]] for a in res.a]
    res = res[["side", "target", "mean_a", "mean_b", "diff", "p"]].rename(columns={"mean_a": "pre", "mean_b": "post"})
    return tab, res, k, dfe

tabR, resR, kR, dfR = prepost_tests(seedNT, TERMS4, ["mk"], 17, ROI)
netNT = network_table(seedNT)
tabN, resN, kN, dfN = prepost_tests(netNT, TERMS4, ["mk"], 6, dict(enumerate(NET, start=1)))
print("\nROI model, N and T:"); print(tabR[["df", "F", "PR(>F)", "eta_p2"]].to_string())
print("Tukey-Kramer over", kR, "cells, error df", dfR)
print(resR.to_string())
print("\nNetwork model, N and T:"); print(tabN[["df", "F", "PR(>F)", "eta_p2"]].to_string())
print(resN.to_string())
print("\nscipy's studentized range gives p down to about 1.4e-11; smaller p are printed at that floor.")

## fig5a-pre-scc-connectivity

> Before stimulation, the SCC had positive connectivity with vmFC, dACC, PCC, STG, hippocampus,
> amygdala and DLPFC, and negative connectivity with the insula and somatosensory cortex.

Per target ROI, the mean normalized z of the pre-DBS sessions over voxels pooled across N and T;
reading (a) the stimulated hemisphere, (b) both hemispheres.

In [ ]:
pre = seedNT[seedNT.pp == 1]
pos_rois = ["vmFC", "dACC", "PCC", "STG", "HIP", "AMY", "DLPFC"]
neg_rois = ["insula", "sensory"]
inv = {v: k for k, v in ROI.items()}
out = []
for reading, sel in [("a stim hemisphere", pre[pre.side == 1]), ("b both hemispheres", pre)]:
    m = sel.groupby("roi").z.mean()
    for r in pos_rois + neg_rois:
        want = ">0" if r in pos_rois else "<0"
        v = m[inv[r]]
        out.append(dict(reading=reading, roi=r, mean_z=v, expected=want, holds=(v > 0) if want == ">0" else (v < 0)))
pre_tab = pd.DataFrame(out)
print(pre_tab.to_string())
print(pre_tab.groupby("reading").holds.sum())
# by animal
print(pre[pre.side == 1].groupby(["roi", "mk"]).z.mean().unstack("mk").rename(index=ROI).loc[pos_rois + neg_rois].rename(columns=code_of))

**Result: reproduced.** Under both readings the seven named ROIs have a positive mean
normalized z before stimulation and the anterior-dorsal insula and sensory cortex a negative one
(9 of 9 signs, table above).

## fig5b-scc-decreases

> After SCC-DBS, connectivity of the stimulated SCC fell with PCC, vmFC, STG, hippocampus,
> amygdala, DLPFC and dACC (PrePost x side interaction F(1,20279) = 186.48, p = 2.87e-42;
> Tukey-Kramer p < 0.001 in the stimulated hemisphere).

Model of the code (Fig5BCandE9_dbs_FC_25norm_mkNT_analysisscript_a.m): type III anovan on monkey,
PrePost, side and ROI, Tukey-Kramer over the 68 PrePost x side x ROI cells.

In [ ]:
r = tabR.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfR)}) = {r['F']:.2f}, p = {r['PR(>F)']:.3g}, eta_p2 = {r['eta_p2']:.4f}")
dec = ["PCC", "vmFC", "STG", "HIP", "AMY", "DLPFC", "dACC"]
s = resR[(resR.side == "stim") & resR.target.isin(dec)]
print(s.to_string())
print("ROIs with p < 0.05 and post below pre:", int(((s.p < 0.05) & (s["diff"] < 0)).sum()), "of 7")
print("voxels per side (expected PCC 249, vmFC 109, STG 499, DLPFC 766):", cnt.loc[["PCC", "vmFC", "STG", "DLPFC"], "stim side"].to_dict())
print("all 17 ROIs, stim and control side voxel counts:"); print(cnt.T.to_string())

**Result: reproduced.** The PrePost x side interaction is F(1,20279) = 186.48, p = 2.9e-42, as
reported, and all seven ROIs fall in the stimulated hemisphere with Tukey-Kramer p below 0.001.
The voxel counts are those of Supplementary Table 8 (PCC 249, vmFC 109, STG 499, DLPFC 766), as
they come from the paper's own voxel lists.

## fig5b-scc-increases

> Connectivity of the stimulated SCC with the anterior-dorsal insula and somatosensory cortex
> increased after SCC-DBS (Tukey-Kramer p < 0.001).

In [ ]:
s = resR[(resR.side == "stim") & resR.target.isin(["insula", "sensory"])]
print(s.to_string())

**Result: reproduced.** Both ROIs rise in the stimulated hemisphere (insula -0.013 to 0.026,
sensory cortex -0.026 to 0.006) with Tukey-Kramer p below 1e-7.

## ed7-decreases-both-monkeys

> The reductions in connectivity from the SCC were consistent across both stimulated monkeys.

In [ ]:
st = seedNT[seedNT.side == 1]
per = st.groupby(["mk", "roi", "pp"]).z.mean().unstack("pp")
per["post_minus_pre"] = per[2] - per[1]
tab7 = per.post_minus_pre.unstack("mk").rename(index=ROI, columns=code_of).loc[dec]
print(tab7.to_string())
print("ROIs below 0 in both monkeys:", int((tab7 < 0).all(axis=1).sum()), "of 7")
print("PCC and vmFC below 0 in both:", bool((tab7.loc[["PCC", "vmFC"]] < 0).all().all()))

**Result: reproduced.** In the stimulated hemisphere all seven ROIs have a lower mean normalized z after DBS in both N and T (changes -0.0006 to -0.103); the smallest decrease is the amygdala of N (-0.0006).

## fig5-bilateral-changes

> Although stimulation was unilateral, connectivity changes with the stimulated SCC were also
> present in the unstimulated control hemisphere.

In [ ]:
c = resR[resR.side == "control"]
print(c.to_string())
hubs = ["vmFC", "PCC", "insula", "dACC"]
print("hubs with p < 0.05 on the control side:", int((c[c.target.isin(hubs)].p < 0.05).sum()), "of 4")
print("control-side ROIs with p < 0.05:", int((c.p < 0.05).sum()), "of 17 (Supplementary Table 9: 11)",
      sorted(c.target[c.p < 0.05]))

**Result: reproduced.** All four hubs (vmFC, PCC, anterior-dorsal insula, dACC) change on the
control side at Tukey-Kramer p < 0.05 (insula p = 0.029, the others below 1e-6). Eleven of 17
control-side ROIs are significant, the same eleven as Supplementary Table 9.

## fig5c-network-decreases

> After SCC-DBS, connectivity between the stimulated SCC and the DMN, LIM and CEN hubs decreased
> significantly (PrePost x side interaction F(1,21967) = 51.09, p = 9.10e-13; Tukey-Kramer p < 0.001).

Networks as in the code (LIM includes area 25); the Methods' LIM without area 25 is a variant.

In [ ]:
r = tabN.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfN)}) = {r['F']:.2f}, p = {r['PR(>F)']:.3g}, eta_p2 = {r['eta_p2']:.4f}")
print(resN.to_string())
ncnt = netNT[(netNT.pp == 1)].groupby(["roi", "side"]).size().unstack("side")
ncnt.index = list(NET)
print("voxels per network and side (expected DMN 857, LIM 821, CEN 1102):"); print(ncnt.T.to_string())
# variant: LIM without area 25 (Methods)
NET_M = dict(NET, LIM=[1, 5, 9, 10, 11, 12])
parts = []
for i, (name, rois) in enumerate(NET_M.items(), start=1):
    s_ = seedNT[seedNT.roi.isin(rois)].copy(); s_["roi"] = i; parts.append(s_)
netM = pd.concat(parts, ignore_index=True)
tabM, resM, _, dfM = prepost_tests(netM, TERMS4, ["mk"], 6, dict(enumerate(NET_M, start=1)))
print("variant, LIM without area 25: PrePost x side F =", round(tabM.loc["pp x side", "F"], 2), "error df", int(dfM))
print(resM[resM.side == "stim"].to_string())

**Result: reproduced.** F(1,21967) = 51.09, p = 9.1e-13, as reported; the stimulated-side DMN,
LIM and CEN fall with Tukey-Kramer p at the floor of 1e-11. With the Methods' LIM (no area 25)
the LIM still falls (interaction F = 61.6; the Tukey-Kramer p is printed as 0 with scipy's warning that the integral is probably divergent, so it could not be computed). Voxel counts are DMN 857, LIM 821, CEN 1102.

## fig5c-smn-increase

> Connectivity between the SCC and the sensorimotor network also changed after SCC-DBS, as an
> increase.

In [ ]:
print(resN[resN.target == "SMN"].to_string())

**Result: reproduced.** The stimulated-side SMN rises (post above pre, Tukey-Kramer p at the floor
of 1e-11; the paper reports 2.83e-20).

## disc-no-motor-roi-change

> Connectivity between the stimulated SCC and the primary motor cortex or the supplementary motor
> area did not change after SCC-DBS.

In [ ]:
print(resR[resR.target.isin(["motor", "SMA"])].to_string())

**Result: reproduced.** On the stimulated side motor cortex (change +0.006, p = 0.9999) and SMA
(change -0.002, p = 1.0) do not change; on the control side neither does (p = 0.999 and 1.0).

## fig5c-vis-no-change

> Connectivity between the SCC and the visual network, or areas within it, did not change.

In [ ]:
print(resN[(resN.target == "VIS")].to_string())
print(resR[resR.target.isin(["V1", "MT"])].to_string())

**Result: reproduced.** On the stimulated side VIS (network model, p = 1.0), V1 (p = 1.0) and MT (p = 0.999) do not change. On the control side MT falls (p = 1.0e-06), as Supplementary Table 9 reports, and VIS does not (p = 0.65).

## fig5bc-san-no-net-change

> The salience network as a whole did not change its connectivity with the SCC, because its nodes
> changed in opposite directions.

In [ ]:
print(resN[resN.target == "SAN"].to_string())
print(resR[(resR.side == "stim") & resR.target.isin(["insula", "dACC"])].to_string())

**Result: reproduced.** The stimulated-side SAN does not change (+0.0002, p = 1.0) while its nodes move apart: the anterior-dorsal insula rises (p = 6.8e-08) and the dACC falls (p = 4.9e-04). On the control side the SAN falls (p = 6.9e-07).

## fig6d-sham-scc-little-change

> After lead implantation without stimulation, connectivity of the SCC changed little; among the
> DMN and SAN hubs only the sham-side vmFC and control-side anterior-dorsal insula changed
> (PrePost x side F(1,10106) = 4.58, p = 0.0323).

Model of Fig6DandE10D_dbs_FC_25norm_mkD_analysisscript_a.m: the full three-way factorial on monkey
D, no monkey term.

In [ ]:
tabD, resD, kD, dfD = prepost_tests(seedD, TERMS3, [], 17, ROI)
r = tabD.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfD)}) = {r['F']:.2f}, p = {r['PR(>F)']:.3g}, eta_p2 = {r['eta_p2']:.5f}")
hubs = ["vmFC", "PCC", "insula", "dACC"]
sD = resD[resD.target.isin(hubs)]
print(sD.to_string())
print("significant of the 8 hub comparisons:", int((sD.p < 0.05).sum()))
print("sham-side PCC and dACC significant decreases:",
      int(((sD.side == "stim") & sD.target.isin(["PCC", "dACC"]) & (sD.p < 0.05) & (sD["diff"] < 0)).sum()))
print("all 17 ROIs:"); print(resD.to_string())

# The local increase at the lead site. The criterion reads it from the map thresholded as in
# fig5a-control-group-no-reduction; with one map per session for D there is no voxel test (neither
# runs nor animals to test across), so the unthresholded change is shown, reported and not judged.
pre_d = np.asarray(h.fc_map("D", "pre").dataobj, dtype=float)
post_d = np.asarray(h.fc_map("D", "post").dataobj, dtype=float)
brain = (pre_d != 0) & (post_d != 0)
chg = np.where(brain, post_d - pre_d, np.nan)
a25 = resD[resD.target == "area25"]
print("area 25 (seed area; sham side = implanted, left), pre to post:"); print(a25.to_string())
imgD = h.fc_map("D", "pre")
top = np.argsort(np.nan_to_num(chg, nan=-np.inf).ravel())[::-1][:10]
ijk = np.array(np.unravel_index(top, chg.shape)).T
xyz = (imgD.affine @ np.c_[ijk, np.ones(len(ijk))].T).T[:, :3]
lab = {}
srcD = fc_src["D"].dropna(subset=["fc", "subject"])
lij, _ = h.rai_to_index(imgD, srcD.x, srcD.y, srcD.z)
for (i, j, k), r_, sd in zip(lij, srcD.roi.astype(int), srcD.side.astype(int)):
    lab[(i, j, k)] = f"{ROI[r_]} {'sham' if sd == 1 else 'control'} side"
print("the ten largest increases in D (RAS mm, NMT grid) and the ROI list they fall in:")
for (i, j, k), (x, y, z) in zip(ijk, xyz):
    print(f"  ({x:6.1f}, {y:6.1f}, {z:6.1f})  change {chg[i, j, k]:+.3f}  {lab.get((i, j, k), 'no ROI')}")
print("mean change over the brain:", round(float(np.nanmean(chg)), 4))

**Result: reproduced.** In monkey D the PrePost x side interaction is F(1,10106) = 4.58, p = 0.032, as reported. Two of the eight hub comparisons are significant, the sham-side vmFC (a decrease, p = 8.0e-04) and the control-side insula (an increase, p = 1.1e-11), as the paper says, and the sham-side PCC and dACC do not change (p = 1.0 and 1.0). Outside the hubs some ROIs do change, among them the sham-side area 25, the seed region, which rises (+0.18, p = 4.3e-12). The local increase at the lead site, which the criterion reports and does not judge, cannot be read from a thresholded map: D has one map per session, so there is no voxel test. In the unthresholded change the ten largest increases (+0.17 to +0.30) all lie at the midline next to the sham-side area 25, six of them inside its voxels and one in the sham-side vmFC, while the mean change over the brain is +0.0008.

## fig6e-sham-no-network-reduction

> After lead implantation without stimulation, the reductions in connectivity between the SCC and
> the DMN, LIM and CEN seen after SCC-DBS did not appear (PrePost x side F(1,10972) = 7.36, p = 0.0067).

In [ ]:
netD = network_table(seedD)
tabDN, resDN, _, dfDN = prepost_tests(netD, TERMS3, [], 6, dict(enumerate(NET, start=1)))
r = tabDN.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfDN)}) = {r['F']:.2f}, p = {r['PR(>F)']:.3g}")
print(resDN.to_string())
dcnt = netD[netD.pp == 1].groupby(["roi", "side"]).size().unstack("side"); dcnt.index = list(NET)
print("DMN voxels sham side, control side (expected 453, 404):", dcnt.loc["DMN"].tolist())
s = resDN[(resDN.side == "stim") & resDN.target.isin(["DMN", "LIM", "CEN"])]
print("sham-side DMN, LIM, CEN with a significant decrease:", int(((s.p < 0.05) & (s["diff"] < 0)).sum()))

**Result: reproduced.** F(1,10972) = 7.36, p = 0.0067, as reported. On the sham side DMN, LIM and CEN do not change (p = 0.73, 1.0 and 0.76); the sham-side SMN rises (p = 1.4e-04). On the control side the DMN rises (p = 1.0e-07), the SAN rises (p = 1.1e-04) and the CEN falls (p = 0.0089).

## fig5a-control-group-no-reduction

> The reductions in SCC connectivity, and bilateral changes, were absent in the nonsurgical control
> group over a comparable interval.

Blocked: reading (a) of the criterion, a two-sample test across runs, needs per-run seed maps,
and the dandiset holds one map per animal and session. Reading (b), a one-sample t-test of the
per-animal change (B, W, H; df 2), is run below at the voxel threshold of the Fig. 5a legend (two-sided
P < 0.01) and the cluster size (above 20 voxels, faces touching), without the cluster-level
correction (AFNI 3dClustSim), which is not in this environment. Without it any cluster found is
counted, so a test with the correction finds no more clusters than this one. PCC and vmFC are
the voxels of the paper's ROI lists for N and T (link-fc-voxels), on the same NMT grid, in both
hemispheres.


In [ ]:
from scipy import stats as st_
from scipy import ndimage
ctrl = {mk: {s: np.asarray(h.fc_map(mk, s).dataobj, dtype=float) for s in ("pre", "post")} for mk in "BWH"}
print({mk: v["pre"].shape for mk, v in ctrl.items()}, "grid of N's map:", h.fc_map("N", "pre").shape)
mask = np.all([v[s] != 0 for v in ctrl.values() for s in v], axis=0)
dchg = np.stack([v["post"] - v["pre"] for v in ctrl.values()])
mean_c = dchg.mean(0)
tval = np.where(mask, mean_c / (dchg.std(0, ddof=1) / np.sqrt(3)), 0)
tcrit = st_.t.ppf(1 - 0.005, 2)
print("voxels in the mask:", int(mask.sum()), "; |t| threshold for two-sided P < 0.01, df 2:", round(tcrit, 2))
img0 = h.fc_map("N", "pre")
srcNT = fc_src["NT"].dropna(subset=["fc", "subject"])
rij, _ = h.rai_to_index(img0, srcNT.x, srcNT.y, srcNT.z)
roi_mask = {}
for name, code in [("PCC", 1), ("vmFC", 2)]:
    mm = np.zeros(mask.shape, bool)
    sel = rij[srcNT.roi.values.astype(int) == code]
    mm[tuple(sel.T)] = True
    roi_mask[name] = mm
    print(name, "voxels (both hemispheres, N and T lists pooled):", int(mm.sum()))
for sign, name in [(-1, "negative"), (1, "positive")]:
    supra = (sign * tval > tcrit) & mask
    lab_, n_ = ndimage.label(supra)          # faces touching (AFNI NN1)
    sizes = np.bincount(lab_.ravel())[1:]
    big = [i + 1 for i, s_ in enumerate(sizes) if s_ > 20]
    print(f"{name}: {int(supra.sum())} voxels above threshold, {n_} clusters, largest {int(sizes.max()) if n_ else 0} voxels; above 20 voxels: {len(big)}")
    for b in big:
        cl = lab_ == b
        print("   cluster of", int(cl.sum()), "voxels; overlaps PCC", int((cl & roi_mask["PCC"]).sum()), ", vmFC", int((cl & roi_mask["vmFC"]).sum()))
print("per-animal mean change (post minus pre) in the PCC and vmFC voxels:")
print(pd.DataFrame({name: [float(dchg[i][mm & mask].mean()) for i in range(3)] for name, mm in roi_mask.items()}, index=list("BWH")).round(4).to_string())


**Result: blocked (data missing).** Reading (a) cannot be run, and the criterion blocks the claim
unless both readings are; reading (b), uncorrected, is shown above for the record, with the numbers
there.


## link-fa-voxels

> The tract voxels of the paper's FA source data (DANDI:001773) can be located in the dandiset's FA
> maps, so that the tract and subregion labels of the source data can be applied to the dandiset's
> FA values.

The source data list FA values without positions; each value is looked up in the map of its
monkey and session (exact float32 equality, else the nearest value within 1e-6).

In [ ]:
fl = fa_link.copy()
fl["frac_matched"] = fl.matched / fl["values"]
print(fl.to_string())
print("overall matched:", int(fl.matched.sum()), "of", int(fl["values"].sum()),
      f"({fl.matched.sum() / fl['values'].sum():.4f}); ambiguous:", int(fl.ambiguous.sum()))
print("by path: exact", int(fl.exact.sum()), "; nearest within 1e-6", int(fl.nearest.sum()),
      "; ambiguous among the exact", int((fl.ambiguous - fl.ambiguous_nearest).sum()),
      "; ambiguous among the nearest (more than one voxel within 1e-6)", int(fl.ambiguous_nearest.sum()))
# For an ambiguous value, the voxels it could be hold values within 2e-6 of each other, so the
# FA used does not depend on which one is taken; the spread of the candidates is shown.
spread = []
for key, t in fa_src.items():
    for (mk, pp), s in t[t.nmatch > 1].groupby(["mk", "prepost"]):
        flat = fa_maps[(code_of[mk], "pre" if pp == 1 else "post")].ravel()
        for v in s.FA.values:
            c = flat[np.abs(flat - v) < 1e-6]
            spread.append(c.max() - c.min())
print("largest spread of candidate values for an ambiguous match:", float(max(spread)) if spread else 0.0)

**Result: the link holds.** All 6332 FA values of the four source sheets are found in the map of their monkey and session, within 5e-7 (the spreadsheets' rounding): 199 by exact float32 equality and 6133 as the nearest value within 1e-6. 641 values have more than one candidate voxel (17 exact, 624 within 1e-6), so for about one value in ten the voxel itself is not identified; the candidates' values differ by at most 1.6e-6, so the FA used does not depend on the choice, and the tests below use values only. The lists repeat voxels in two ways (data checks above). Within one monkey, session, side and tract there are 99 repeats in the N and T whole-tract list, 44 in the subregion list, and 40 and 18 in the lists of D, and each is the same value listed twice in the paper's sheet. Across the tracts of one session, 233 voxels of the N and T whole-tract list are listed in two or more tracts (2 of them across sides), most often UF with FM or CB with UF on the same side, as are 71 voxels of the whole-tract list of D, 1 of the N and T subregion list and 18 of the D subregion list (dACC-CB with MCC-CB). Each of these carries the same value in every tract it is listed in, and no voxel is matched by two different values. For 23, 12, 1 and 3 of them the value has more than one candidate voxel, so they may be two voxels of equal value; the others are one voxel in two tract lists, where the paper's tracts overlap. The paper explains neither kind of repeat. The analysis keeps the lists as the paper used them, repeats included, so the voxel n are the paper's. Since the dandiset's values equal the paper's and every voxel selection (tracts, lead-track exclusion, CB subregions) is the paper's, the FA claims below re-run the paper's statistics on its own data; they show that the dandiset holds the data behind them, not that the selections would be found again.

In [ ]:
def fa_tests(t, mkD, names):
    df = t[t.voxel >= 0].rename(columns={"prepost": "pp", "part": "roi", "FA_ds": "z"})[["mk", "pp", "side", "roi", "z"]].copy()
    terms = TERMS3 if mkD else TERMS4
    return prepost_tests(df, terms, [] if mkD else ["mk"], len(names), names), df

TRACT = {1: "CB", 2: "UF", 3: "FM"}
PART = {1: "SCC-CB", 2: "dACC-CB", 3: "MCC-CB", 4: "PCC-CB"}
(tabFA, resFA, kFA, dfFA), dfFAw = fa_tests(fa_src["whole_NT"], False, TRACT)
(tabFS, resFS, kFS, dfFS), dfFAs = fa_tests(fa_src["sub_NT"], False, PART)
(tabFAD, resFAD, _, dfFAD), dfFADw = fa_tests(fa_src["whole_D"], True, TRACT)
(tabFSD, resFSD, _, dfFSD), dfFADs = fa_tests(fa_src["sub_D"], True, PART)
for name, tab, res, dfe in [("whole tract N,T", tabFA, resFA, dfFA), ("CB parts N,T", tabFS, resFS, dfFS),
                            ("whole tract D", tabFAD, resFAD, dfFAD), ("CB parts D", tabFSD, resFSD, dfFSD)]:
    print("\n" + name, "error df", int(dfe)); print(tab[["df", "F", "PR(>F)", "eta_p2"]].to_string()); print(res.to_string())

## fig2a-cb-fa-increase

> After 6 weeks of SCC-DBS, FA increased in the cingulum bundle of the stimulated hemisphere
> (PrePost x side interaction F(1,3187) = 4.64, p = 0.0312; Tukey-Kramer pre against post in the
> stimulated CB, p = 0.022).

FA values from the dandiset at the voxels of the paper's tract lists (link-fa-voxels); model of
Fig2A_dbs_FA_wholetract_analysisscript_a.m. The lead-track exclusion of the claims file's two
readings is the paper's own, already applied in its voxel lists, so both readings give this one
result.

In [ ]:
r = tabFA.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfFA)}) = {r['F']:.2f}, p = {r['PR(>F)']:.4f}, eta_p2 = {r['eta_p2']:.4f}")
print(resFA[(resFA.side == "stim") & (resFA.target == "CB")].to_string())
vc = dfFAw[dfFAw.pp == 1].groupby(["side", "roi"]).size().unstack("roi").rename(columns=TRACT, index={1: "stim", 2: "control"})
print("voxels (expected stim CB 380, UF 252, FM 105; control CB 363, UF 367, FM 133):"); print(vc.to_string())

**Result: reproduced.** PrePost x side F(1,3187) = 4.64, p = 0.031, and the stimulated CB rises from 0.324 to 0.369 (Tukey-Kramer p = 0.022), as reported. The voxel counts equal the expected ones (380, 363, 252, 367, 105, 133), as they come from the paper's lists.

## fig2a-no-other-tract-change

> FA did not change consistently in the stimulated UF or FM, nor in any of the three tracts in the
> control hemisphere.

In [ ]:
o = resFA[~((resFA.side == "stim") & (resFA.target == "CB"))]
print(o.to_string())
print("of these five, p < 0.05:", int((o.p < 0.05).sum()))

**Result: reproduced.** None of the five other pre-post comparisons is significant (Tukey-Kramer p = 0.86 to 1.0; changes -0.037 to +0.009).

## ed2-cb-both-monkeys

> Both stimulated monkeys showed an FA change in the same direction in the stimulated CB.

In [ ]:
s = dfFAw[(dfFAw.side == 1) & (dfFAw.roi == 1)]
e2 = s.groupby(["mk", "pp"]).z.mean().unstack("pp").rename(index=code_of, columns={1: "pre", 2: "post"})
e2["post_minus_pre"] = e2.post - e2.pre
print(e2.to_string())

**Result: reproduced.** Mean stimulated-CB FA rises in both monkeys, by 0.019 in N and 0.079 in T.

## fig2b-mcc-cb-increase

> Of four CB subregions, only the mid-cingulate part on the stimulated side showed an FA increase,
> in both animals (PrePost x side interaction F(1,1357) = 5.42, p = 0.020; Tukey-Kramer for the
> stimulated MCC-CB, p = 0.0017).

Subregions are the paper's own (its voxel lists), so the claims file's two readings of
[CB-PARTS] give this one result.

In [ ]:
r = tabFS.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfFS)}) = {r['F']:.2f}, p = {r['PR(>F)']:.4f}, eta_p2 = {r['eta_p2']:.4f}")
print(resFS.to_string())
others = resFS[~((resFS.side == "stim") & (resFS.target == "MCC-CB"))]
print("other comparisons with a significant increase:", int(((others.p < 0.05) & (others["diff"] > 0)).sum()))
s = dfFAs[(dfFAs.side == 1) & (dfFAs.roi == 3)]
e = s.groupby(["mk", "pp"]).z.mean().unstack("pp").rename(index=code_of, columns={1: "pre", 2: "post"})
e["post_minus_pre"] = e.post - e.pre
print("stimulated MCC-CB per monkey:"); print(e.to_string())
vc = dfFAs[dfFAs.pp == 1].groupby(["side", "roi"]).size().unstack("roi").rename(columns=PART, index={1: "stim", 2: "control"})
print("voxels (expected stim MCC 55, stim other three 271, control four 361):"); print(vc.to_string())
print("stim other three:", int(vc.loc["stim"].drop("MCC-CB").sum()), " control four:", int(vc.loc["control"].sum()))

**Result: reproduced.** PrePost x side F(1,1357) = 5.42, p = 0.020; the stimulated MCC-CB rises from 0.416 to 0.551 (Tukey-Kramer p = 0.0018); none of the seven other comparisons rises significantly (p = 0.91 to 1.0); the stimulated MCC-CB rises in N (+0.081) and T (+0.205). The voxel counts are 55, 271 and 361, as expected.

## fig6a-sham-cb-fa-decrease

> In the nonstimulation control animal, FA decreased in the CB of the implanted hemisphere
> (Tukey-Kramer p = 5.07e-09), with no change in the other tracts.

In [ ]:
r = tabFAD.loc["pp x side x roi"]
print(f"PrePost x side x tract: F({int(r['df'])},{int(dfFAD)}) = {r['F']:.2f}, p = {r['PR(>F)']:.4f}, eta_p2 = {r['eta_p2']:.4f}")
print(resFAD.to_string())
vc = dfFADw[dfFADw.pp == 1].groupby(["side", "roi"]).size().unstack("roi").rename(columns=TRACT, index={1: "sham", 2: "control"})
print("voxels (expected sham CB 162, control CB 148):"); print(vc.to_string())

**Result: reproduced.** In D the sham-side CB falls from 0.362 to 0.245 (Tukey-Kramer p = 8.4e-09; reported 5.07e-09), and none of the other five comparisons is significant (smallest p = 0.088, control-side FM). The three-way interaction is F(2,1238) = 6.02, p = 0.0025, so the printed F 1,1238 is a misprint for 2 df, as the claims file reads it. Voxels 162 and 148, as expected.

## fig6b-sham-dacc-cb-decrease

> In the nonstimulation control animal, the FA decrease lay in the dACC portion of the CB
> (PrePost x side F(1,492) = 24.55, p = 9.97e-07; dACC-CB Tukey-Kramer p = 1.30e-08).

In [ ]:
r = tabFSD.loc["pp x side"]
print(f"PrePost x side: F({int(r['df'])},{int(dfFSD)}) = {r['F']:.2f}, p = {r['PR(>F)']:.3g}, eta_p2 = {r['eta_p2']:.4f}")
print(resFSD.to_string())

**Result: reproduced.** PrePost x side F(1,492) = 24.55, p = 1.0e-06; the sham-side dACC-CB falls from 0.413 to 0.230 (Tukey-Kramer p = 3.6e-08; reported 1.30e-08), and the sham-side MCC-CB does not rise (it falls, p = 0.44).

## link-if-counts

> Cell counts computed from the dandiset's immunofluorescence images agree, field by field, with
> the CC1 and DAPI counts of the paper's source data (DANDI:001773).

Counting (helper `count_cells`): one threshold per section and channel over all fields of the
section (Methods: "section-specific intensity thresholds for each channel"), objects of 8-connected
pixels above it, no watershed. Readings: threshold IsoData (FIJI's default) or Otsu (claims file,
[IF]); smallest object 10 or 50 pixels (not given by the paper; an assumption, shown both ways).
Fields are matched to the sheets by section, hemisphere and field number in the file name.
Criterion (written for this link claim): the link holds in a region when, under at least one
counting reading, the per-field counts correlate with the paper's at r of at least 0.7 for both
CC1 and DAPI and the median ratio of counts is within 0.8 to 1.25; reproduced when it holds in
all four regions, partially reproduced in one to three, not reproduced in none.

In [ ]:
READINGS = [(m, a) for m in ("isodata", "otsu") for a in (10, 50)]
CH = {"CC1": 0, "DAPI": 2}
counts = []
for (region, section), g in IF.groupby(["region", "section"]):
    ims = {p: h.load_tiff(p) for p in g.path}
    for name, c in CH.items():
        pix = np.concatenate([im[..., c].ravel() for im in ims.values()])
        for meth, amin in READINGS:
            t = h.section_threshold(pix, meth)
            for p, im in ims.items():
                k = g.set_index("path").loc[p]
                counts.append(dict(region=region, section=section, hemi=k.hemi, field=k.field, mark=k["mark"],
                                   channel=name, method=meth, min_area=amin, threshold=t,
                                   count=h.count_cells(im[..., c], t, amin)))
    del ims
counts = pd.DataFrame(counts)
cw = counts.pivot_table(index=["region", "section", "hemi", "field", "mark", "method", "min_area"],
                        columns="channel", values="count").reset_index()
cw["ratio"] = 100 * cw.CC1 / cw.DAPI
cw["monkey"] = cw.section.str[0]
cw["side"] = np.where(cw.hemi == cw.monkey.map(h.STIM_HEMI), 1, 2)
print(len(counts), "counts;", cw.groupby(["region", "method", "min_area"]).size().unstack(["method", "min_area"]))

In [ ]:
# The paper's per-field sheets.
def if_sheet(name):
    t = h.source_sheet(name).iloc[:, :7]
    t.columns = ["CC1", "DAPI", "ratio", "mk", "side", "section", "field"]
    t = t.dropna(subset=["CC1"]).copy()
    t["section"] = t.section.astype(str).str.strip()
    for c in ["mk", "side", "field"]:
        t[c] = t[c].astype(int)
    return t

sheets = {"MCC_CB_mkNT": if_sheet("CC1_MCC_CB_data_a.xlsx"), "dACC_CB_mkNT": if_sheet("CC1_dACC_CB_data_a.xlsx"),
          "UF_FM_overlap_mkNT": if_sheet("CC1_UF-FMoverlap_data_a.xlsx")}
d_raw = h.source_sheet("CC1_mkD_MCC_CB_data_a.xlsx").dropna(subset=["slide"])
dsheet = pd.concat([pd.DataFrame(dict(CC1=d_raw.CC1_leadside, DAPI=d_raw.DAPI_leadside, ratio=d_raw.ratio_leadside, side=1)),
                    pd.DataFrame(dict(CC1=d_raw.CC1_controlside, DAPI=d_raw.DAPI_controlside, ratio=d_raw.ratio_controlside, side=2))])
dsheet["section"] = list(d_raw.slide.astype(str).str.strip()) * 2
dsheet["field"] = list(d_raw.location.astype(int)) * 2
dsheet["mk"] = 3
sheets["MCC_CB_mkD"] = dsheet
for k, s in sheets.items():
    print(k, len(s), "fields;", s.groupby("side").size().to_dict())

link = []
merged = {}
for region, s in sheets.items():
    c = cw[cw.region == region]
    m = c.merge(s, on=["section", "side", "field"], how="outer", suffixes=("", "_paper"), indicator=True)
    merged[region] = m
    for (meth, amin), g in m[m._merge == "both"].groupby(["method", "min_area"]):
        link.append(dict(region=region, method=meth, min_area=int(amin), fields=len(g),
                         CC1_r=np.corrcoef(g.CC1, g.CC1_paper)[0, 1], DAPI_r=np.corrcoef(g.DAPI, g.DAPI_paper)[0, 1],
                         ratio_r=np.corrcoef(g.ratio, g.ratio_paper)[0, 1],
                         CC1_med_rel=np.median(g.CC1 / g.CC1_paper), DAPI_med_rel=np.median(g.DAPI / g.DAPI_paper),
                         ratio_med_diff=np.median(g.ratio - g.ratio_paper)))
    only_img = m[m._merge == "left_only"][["section", "hemi", "field", "mark"]].drop_duplicates()
    only_sheet = m[m._merge == "right_only"][["section", "side", "field"]].drop_duplicates()
    print(region, "images without a sheet row:", only_img.values.tolist(), "; sheet rows without an image:", only_sheet.values.tolist())
link = pd.DataFrame(link)
print(link.round(3).to_string())

In [ ]:
# The criterion region by region: the readings under which the link holds.
link["holds"] = ((link.CC1_r >= 0.7) & (link.DAPI_r >= 0.7) & link.CC1_med_rel.between(0.8, 1.25)
                 & link.DAPI_med_rel.between(0.8, 1.25))
print(link[["region", "method", "min_area", "CC1_r", "DAPI_r", "CC1_med_rel", "DAPI_med_rel", "holds"]].round(3).to_string())
held = link.groupby("region").holds.any()
print("regions where the link holds under at least one reading:", int(held.sum()), "of 4;", held.to_dict())
every = link.groupby(["method", "min_area"]).holds.all()
print("readings under which it holds in every region:", [k for k, v in every.items() if v] or "none")

# Channel assignment: the red-channel count against the sheet's CC1 and DAPI columns, and the
# blue-channel count likewise (IsoData, min 10). The levels decide: CC1 cells are a fraction of DAPI cells.
rows = []
for region, m in merged.items():
    g = m[(m._merge == "both") & (m.method == "isodata") & (m.min_area == 10)]
    rows.append(dict(region=region, red_vs_sheet_CC1=np.corrcoef(g.CC1, g.CC1_paper)[0, 1],
                     red_vs_sheet_DAPI=np.corrcoef(g.CC1, g.DAPI_paper)[0, 1],
                     blue_vs_sheet_DAPI=np.corrcoef(g.DAPI, g.DAPI_paper)[0, 1],
                     blue_vs_sheet_CC1=np.corrcoef(g.DAPI, g.CC1_paper)[0, 1],
                     median_red=g.CC1.median(), median_blue=g.DAPI.median(),
                     median_sheet_CC1=g.CC1_paper.median(), median_sheet_DAPI=g.DAPI_paper.median()))
print(pd.DataFrame(rows).round(3).to_string())
ex = merged["MCC_CB_mkNT"]
ex = ex[(ex._merge == "both") & (ex.method == "isodata") & (ex.min_area == 10)].iloc[0]
print("one field,", ex.section, ex.hemi, ex.field, ": red", int(ex.CC1), "blue", int(ex.DAPI),
      "; sheet CC1", int(ex.CC1_paper), "DAPI", int(ex.DAPI_paper))


In [ ]:
# Per-field counts made here against the paper's, by region and channel (IsoData; the two size filters).
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for j, (region, m) in enumerate(merged.items()):
    for i, ch in enumerate(["CC1", "DAPI"]):
        ax = axes[i, j]
        for amin, colour, marker in [(10, "#2a78d6", "o"), (50, "#eb6834", "s")]:
            g = m[(m._merge == "both") & (m.method == "isodata") & (m.min_area == amin)]
            ax.scatter(g[ch + "_paper"], g[ch], s=14, c=colour, marker=marker, alpha=0.7,
                       label=f"min {amin} px", linewidths=0)
        lim = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lim, lim, color="#888888", lw=1, zorder=0)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_title(f"{region}, {ch}", fontsize=10)
        ax.set_xlabel("paper's count (source data)", fontsize=9)
        if j == 0:
            ax.set_ylabel("count here (IsoData)", fontsize=9)
        ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(frameon=False, fontsize=9)
fig.tight_layout()
plt.show()


**Result: partially reproduced.** Every one of the 360 rows of the paper's four sheets has its image, matched by section, hemisphere and field, and one image (N122 RW6, UF-FM overlap) has no row. By the criterion (r of at least 0.7 for both channels and median ratios within 0.8 to 1.25, under at least one counting reading) the link holds in three of the four regions, and under no single reading in all of them. With the smallest object 10 pixels it holds in the MCC-CB of N and T (r = 0.93 for CC1 and 0.79 for DAPI; median ratio to the paper's counts 1.01 and 0.84) and in the dACC-CB (r = 0.98 and 0.98; ratios 1.14 and 1.08); with 50 pixels it holds in the MCC-CB of D (r = 0.74 and 0.81; ratios 0.88 and 0.90), where 10 pixels gives r = 0.66 for CC1 and a CC1 ratio of 1.30. In the UF-FM overlap it holds under no reading: at 10 pixels the CC1 r is 0.69, at 50 pixels the CC1 ratio is 0.755. The threshold method (IsoData or Otsu) changes none of these by more than 0.01. The red channel is CC1 and the blue DAPI: red counts follow the sheet's CC1 column (r = 0.66 to 0.98) and lie below the blue counts, as CC1-positive cells are a subset of DAPI-positive cells. The criterion of this link claim was written after the counts were first computed. The counting here is an approximation of the paper's FIJI counts (one automatic threshold per section and channel, 8-connected objects, no background subtraction or watershed, a size filter the paper does not give), so the IF results below rest on a pipeline that matches the paper's counts only in part.

In [ ]:
def if_test(region, meth, amin, extra=False):
    # The fields of the paper's design, those with a row of its sheet (Methods, Immunofluorescence:
    # "MCC-CB and dACC-CB: 28 images per hemisphere per animal, 7 fields x 4 sections; overlapped
    # region of UF and FM: 20 images per hemisphere per animal, 5 fields x 4 sections"); with
    # extra=True, images without a row are added (N122 RW6 only).
    m = merged[region]
    keep = m._merge != "right_only" if extra else m._merge == "both"
    g = m[keep & (m.method == meth) & (m.min_area == amin)].copy()
    g["mk"] = g.monkey.map({"N": 1, "T": 2})
    out = {}
    for y in ["ratio", "CC1", "DAPI"]:
        tab, _ = h.anovan(g.rename(columns={y: "y"}), "y", ["mk", "side"], [("mk",), ("side",), ("mk", "side")])
        r = tab.loc["side"]
        means = g.groupby(["monkey", "side"])[y].mean().unstack("side")
        out[y] = dict(F=r.F, df_err=int(tab.loc["Residual", "df"]), p=r["PR(>F)"], eta_p2=r.eta_p2,
                      stim_minus_control=g[g.side == 1][y].mean() - g[g.side == 2][y].mean(),
                      N_stim_minus_control=means.loc["N", 1] - means.loc["N", 2],
                      T_stim_minus_control=means.loc["T", 1] - means.loc["T", 2])
    return pd.DataFrame(out).T

def if_all(region):
    rows = []
    for meth, amin in READINGS:
        t = if_test(region, meth, amin)
        t["reading"] = f"{meth}, min {amin}"
        rows.append(t)
    # the paper's own counts, through the link (not a reading; shown for comparison)
    s = sheets[region].copy()
    s["monkey"] = s.section.str[0]
    out = {}
    for y in ["ratio", "CC1", "DAPI"]:
        tab, _ = h.anovan(s.rename(columns={y: "y"}), "y", ["mk", "side"], [("mk",), ("side",), ("mk", "side")])
        out[y] = dict(F=tab.loc["side", "F"], df_err=int(tab.loc["Residual", "df"]), p=tab.loc["side", "PR(>F)"])
    paper = pd.DataFrame(out).T
    paper["reading"] = "paper's counts (source data)"
    return pd.concat(rows + [paper])

## fig3c-cc1-dapi-ratio-mcc

> The proportion of CC1-positive cells among DAPI-stained cells was higher in the MCC-CB of the
> stimulated hemisphere than in the control hemisphere, in both animals (F(1,108) = 9, p = 0.0034).

Test of Fig3CandE5_dbs_IF_CC1_MCC_CB_analysisscript_a.m: anovan, full model monkey x side, side
effect.

In [ ]:
r3 = if_all("MCC_CB_mkNT")
print(r3.to_string(float_format=lambda v: f"{v:.4g}"))
m = merged["MCC_CB_mkNT"]
print("fields per hemisphere (expected 56):", m[(m.method == "isodata") & (m.min_area == 10)].groupby("side").size().to_dict())

**Result: reproduced.** Under all four counting readings the stimulated MCC-CB has the higher CC1/DAPI ratio (side F(1,108) = 9.5 to 14.4, p = 0.0002 to 0.0026; stimulated minus control 5.1 to 6.9 points), higher in N (5.1 to 8.1) and in T (5.0 to 5.8). With the paper's counts the same test gives F = 9.00, p = 0.0034, as reported. 56 fields per hemisphere.

## ed5-cc1-count-dapi

> In the MCC-CB the number of CC1-positive cells was higher on the stimulated side (F(1,108) = 6.22,
> p = 0.0142), while the number of DAPI-positive cells did not differ (F(1,108) = 0.55, p = 0.4617).

In [ ]:
print(r3.loc[["CC1", "DAPI"]].to_string(float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** CC1 counts are higher on the stimulated side under all four readings (F(1,108) = 5.9 to 9.2, p = 0.0030 to 0.017), and DAPI counts do not differ (p = 0.39 to 0.97).

## ed6a-dacc-cb-no-change

> In the dACC portion of the CB, neither the CC1/DAPI ratio nor the CC1 or DAPI counts differed
> between hemispheres (ratio F(1,108) = 2.93, p = 0.0899).

In [ ]:
r6a = if_all("dACC_CB_mkNT")
print(r6a.to_string(float_format=lambda v: f"{v:.4g}"))

**Result: not reproduced.** Under all four readings the dACC-CB CC1/DAPI ratio differs between hemispheres (F(1,108) = 6.3 to 7.9, p = 0.0058 to 0.013), lower on the stimulated side by 3.4 to 4.0 points; the CC1 and DAPI counts do not differ (p = 0.29 to 0.80). With the paper's own counts the ratio does not differ (F = 2.93, p = 0.090), as reported, so the difference lies in how the cells are counted.

## ed6b-uf-fm-no-change

> In the region where the UF and FM overlap, oligodendrocyte measures did not differ between
> hemispheres (CC1/DAPI ratio F(1,76) = 1.45, p = 0.2319).

The design has 5 fields per section, 20 per hemisphere per animal (Methods, Immunofluorescence of
myelinating oligodendrocytes), and the claims file's n is 40 fields per hemisphere. The image
N122 RW6, a sixth field of that section, has no row in the paper's sheet; the main test uses the
fields of the design (error df 76), and a variant adds N122 RW6.


In [ ]:
r6b = if_all("UF_FM_overlap_mkNT")
print(r6b.to_string(float_format=lambda v: f"{v:.4g}"))
print("smallest p over the four readings and three measures:", float(r6b[r6b.reading != "paper's counts (source data)"].p.min()))
# variant: N122 RW6 added (error df 77)
print("variant with N122 RW6:")
print(pd.concat([if_test("UF_FM_overlap_mkNT", me, a, extra=True).assign(reading=f"{me}, min {a}") for me, a in READINGS]).to_string(float_format=lambda v: f"{v:.4g}"))


**Result: reproduced.** On the fields of the paper's design (40 per hemisphere, error df 76), neither the ratio, the CC1 count nor the DAPI count differs between hemispheres under any of the four counting readings (smallest p = 0.0685, the ratio under Otsu with 50 pixels; with 10 pixels the ratio is at p = 0.28 and 0.28). With the paper's counts the test gives ratio F(1,76) = 1.45, p = 0.23, as reported; its DAPI test gives F = 0.0038, p = 0.9511, where the paper prints 0.9611 (its partial eta squared of 4.97e-05 fits F = 0.0038, so 0.9611 is a misprint). With N122 RW6 added (variant, error df 77), the ratio under Otsu with 50 pixels falls to p = 0.04997, so the outcome would then depend on the size filter; the main test follows the design the Methods state. The counts in this region do not pass the link (link-if-counts).

## fig6c-sham-cc1-no-change

> In the nonstimulation control animal, the CC1/DAPI ratio in the MCC-CB did not differ between the
> implanted and control hemispheres (paired t-test, t(27) = -0.99, p = 0.33).

Paired as rows of the paper's sheet (Fig6C_dbs_IF_CC1_mkD_MCC_CB_analysisscript_a.m), that is by
section and field number.

In [ ]:
from scipy import stats as st_
m = merged["MCC_CB_mkD"]
rows = []
for (meth, amin), g in m[m._merge == "both"].groupby(["method", "min_area"]):
    w = g.pivot_table(index=["section", "field"], columns="side", values="ratio").dropna()
    t, p = st_.ttest_rel(w[1], w[2])
    d = w[1] - w[2]
    rows.append(dict(reading=f"{meth}, min {int(amin)}", pairs=len(w), t=t, p=p, mean_diff=d.mean(), dz=d.mean() / d.std(ddof=1)))
s = sheets["MCC_CB_mkD"].pivot_table(index=["section", "field"], columns="side", values="ratio")
t, p = st_.ttest_rel(s[1], s[2]); d = s[1] - s[2]
rows.append(dict(reading="paper's counts (source data)", pairs=len(s), t=t, p=p, mean_diff=d.mean(), dz=d.mean() / d.std(ddof=1)))
r6c = pd.DataFrame(rows)
print(r6c.to_string(float_format=lambda v: f"{v:.4g}"))
print("readings with p < 0.05:", r6c.reading[(r6c.p < 0.05) & ~r6c.reading.str.startswith("paper")].tolist())
print("the link in the MCC-CB of D, by reading:")
print(link[link.region == "MCC_CB_mkD"][["method", "min_area", "CC1_r", "DAPI_r", "CC1_med_rel", "DAPI_med_rel", "holds"]].round(3).to_string())

**Result: blocked (methods underspecified).** The two size filters give opposite outcomes: with the smallest object 10 pixels the paired t-test gives t(27) = -1.04 (IsoData) and -1.03 (Otsu), p = 0.31, dz = -0.20, close to the paper's t = -0.99, p = 0.33; with 50 pixels t(27) = -2.11 and -2.12, p = 0.044, dz = -0.40, a difference. The paper gives no size filter, and none is defensible over the other: in this region the counts pass the link only at 50 pixels (CC1 median ratio 0.88, r = 0.74; at 10 pixels 1.30 and 0.66), while in the MCC-CB and dACC-CB of N and T they pass only at 10 pixels, and no single reading passes in every region (link-if-counts). Choosing the size per region to fit the paper's counts would not be the paper's procedure, which used one method throughout. The paper's own counts give t = -0.99, p = 0.33.

## ed1-behavior-increase

> In monkeys N and T, time spent moving and foraging in the home cage increased after 6 weeks of
> SCC-DBS (movement F(1,236) = 34.61, foraging F(1,236) = 17.35).

Data elsewhere: the behaviour bins are only in DANDI:001773
(source_data/dbs_behavior_data_a.xlsx). Test of FigE1_dbs_behavior_analysisscript_a.m: anovan, full
model monkey x stim.

In [ ]:
beh = h.source_sheet("dbs_behavior_data_a.xlsx").iloc[:, :6]
print("columns of the sheet, in order:", list(beh.columns))
beh.columns = ["sec", "movement", "foraging", "mk", "stim", "group"]
beh = beh.dropna(subset=["mk"]).copy()
for c in beh.columns:
    beh[c] = beh[c].astype(int)
print(beh.groupby(["mk", "stim", "group"]).size())
nt = beh[beh.mk.isin([1, 2])]
for y in ["movement", "foraging"]:
    tab, _ = h.anovan(nt.rename(columns={y: "y"}), "y", ["mk", "stim"], [("mk",), ("stim",), ("mk", "stim")])
    r = tab.loc["stim"]
    print(f"{y}: stim F({int(r['df'])},{int(tab.loc['Residual', 'df'])}) = {r.F:.2f}, p = {r['PR(>F)']:.3g}")
    print(nt.groupby(["mk", "stim"])[y].mean().unstack("stim").rename(index=code_of, columns={0: "pre", 1: "post"}).to_string())

**Result: reproduced.** Movement F(1,236) = 34.61, p = 1.4e-08 and foraging F(1,236) = 17.35, p = 4.4e-05, as reported, and the post proportion exceeds the pre one in N (movement 0.08 to 0.55, foraging 0.03 to 0.05) and in T (0.20 to 0.38 and 0.02 to 0.30).

## ed1-sham-no-behavior-change

> The nonstimulation control animal (monkey D) showed no change in movement or foraging.

One-way ANOVA of D's 60 pre and 60 post bins (the claims file's criterion).

In [ ]:
dd = beh[beh.mk == 3]
for y in ["movement", "foraging"]:
    a, b = dd[dd.stim == 0][y], dd[dd.stim == 1][y]
    F, p = st_.f_oneway(a, b)
    print(f"{y}: pre {a.mean():.3f} (n={len(a)}), post {b.mean():.3f} (n={len(b)}), F(1,{len(a) + len(b) - 2}) = {F:.2f}, p = {p:.3g}")

**Result: not reproduced.** In monkey D movement does not change (0.38 to 0.27 of bins, F(1,118) = 1.86, p = 0.18), but foraging falls (0.18 to 0.05, F(1,118) = 5.32, p = 0.023), so by the claims file's test one of the two behaviours changes. The paper reports no test for D.

## Summary

| Claim | Outcome |
|---|---|
| link-fc-voxels | reproduced |
| link-fa-voxels | reproduced |
| link-if-counts | partially reproduced |
| ed1-behavior-increase (data elsewhere) | reproduced |
| ed1-sham-no-behavior-change (data elsewhere) | not reproduced |
| fig2a-cb-fa-increase | reproduced |
| fig2a-no-other-tract-change | reproduced |
| ed2-cb-both-monkeys | reproduced |
| fig2b-mcc-cb-increase | reproduced |
| fig3c-cc1-dapi-ratio-mcc | reproduced |
| ed5-cc1-count-dapi | reproduced |
| ed6a-dacc-cb-no-change | not reproduced |
| ed6b-uf-fm-no-change | reproduced |
| fig3f-gratio-decrease | blocked (other: hand measurements) |
| fig4a, fig4b, fig4c, fig4-control, ed10c | blocked (data missing) |
| fig5a-pre-scc-connectivity | reproduced |
| fig5b-scc-decreases | reproduced |
| fig5b-scc-increases | reproduced |
| ed7-decreases-both-monkeys | reproduced |
| fig5-bilateral-changes | reproduced |
| fig5a-control-group-no-reduction | blocked (data missing) |
| fig5c-network-decreases | reproduced |
| fig5c-smn-increase | reproduced |
| disc-no-motor-roi-change | reproduced |
| fig5c-vis-no-change | reproduced |
| fig5bc-san-no-net-change | reproduced |
| fig6a-sham-cb-fa-decrease | reproduced |
| fig6b-sham-dacc-cb-decrease | reproduced |
| fig6c-sham-cc1-no-change | blocked (methods underspecified) |
| fig6d-sham-scc-little-change | reproduced |
| fig6e-sham-no-network-reduction | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The dandiset holds the paper's FA maps and its
normalized seed maps, but neither tract masks nor ROI masks. Which voxel belongs to which tract,
CB subregion or target ROI is taken from the paper's source data (DANDI:001773): by position for
the seed maps and by FA value for the FA maps (link claims). The values analysed are the
dandiset's, and they equal the source data's, so the FA and seed results repeat the paper's
statistics; they show that the dandiset holds the data behind them, not that the masks or ROIs
would be found again. The tract tracking, the registration, the atlas ROIs and the lead-track
exclusion were not redone. Cells were counted here from the images with an automatic threshold
per section and channel, where the paper set thresholds in FIJI by hand; the smallest object
size is not given and is run at 10 and 50 pixels. The counts match the paper's in three of four
regions, under different size filters (10 pixels in the MCC-CB and dACC-CB of N and T, 50 in
the MCC-CB of D), and under no single reading in all four; where the size filter
decides the outcome (fig6c) the claim is blocked. The IF outcomes rest on this approximation of
the paper's counting, and the not reproduced dACC-CB result may come from it: with the paper's
own counts that test is not significant. The p-values of the Tukey-Kramer tests below
about 1e-11 are printed at the floor of scipy's studentized range.

**What was found about the dataset.** It is BIDS-like NIfTI and TIFF, not NWB, and
dataset_description.json is not valid JSON. The FC files are normalized seed maps (one per
session), not EPI data: the connectivity matrices, the per-run analyses and the control-group
voxel test cannot be run as the paper ran it (reading (b) of that test, without the cluster-level
correction, finds no cluster above 20 voxels). No tract or ROI masks are deposited, and nothing in the dandiset says
what the FA and FC files are or in which space they lie. The g-ratio measurements and the cell
counts are not deposited with the images. TR-1-8r.tif duplicates TR-1-8.tif, and the UF-FM folder
has a sixth field (N122 RW6) that the paper's sheet does not use. The paper's FA voxel lists
repeat some voxels.